# DB — Databases (9569)

Revision / content-lookup notebook for **S3B relational design**, **S3C SQL**, **S3D sqlite3**, **S3E NoSQL**, **S3F pymongo**.

Sample files in this folder: `STUDENT.csv`, `LATE.csv`. Mongo needs a local MongoDB on `localhost:27017`.

## Contents
1. [Relational theory](#theory) — keys, normalisation, ER, SQL vs NoSQL
2. [SQLite boilerplates](#sqlite) — connect, CREATE, CSV load, CRUD
3. [SQL query lookup](#sql) — JOIN, GROUP BY, HAVING, LIKE
4. [Design patterns](#design) — composite PK, surrogate key, never-late query
5. [NoSQL + pymongo](#mongo) — prelim-only recall *(parked for A-level P2)*
6. [Exam cheat-sheet](#cheatsheet)

**How to use:** skim markdown for definitions and traps; run code top-down for exam-style patterns. **National P2 = sqlite3 + theory.** pymongo is prelim-only — section 5 is minimum recall, not a drill list.


<a id="theory"></a>
## 1. Relational theory

### Vocabulary
| Term | Meaning |
|------|---------|
| Table / relation | 2D set of records with named fields |
| Record / tuple | one row |
| Field / attribute | one column |
| Primary key (PK) | uniquely identifies a record; not null |
| Foreign key (FK) | field that references another table's PK |
| Composite key | PK made of **two or more** fields together |
| Secondary key | not unique, used for searching (e.g. class, name) |

**PK vs FK trap:** a FK is not a “subset” of a PK — it *references* another table's PK to link rows.

### Normalisation (exam recall)
- **1NF:** atomic values, no repeating groups
- **2NF:** 1NF + no partial dependence on part of a composite PK
- **3NF:** 2NF + no transitive dependence (non-key field depends only on the PK)

JOINs put split-3NF tables back together for a query. They do **not** revert the database to 1NF.

### Keys + 3NF — latecoming sheet (exam pattern)

Unnormalised sheet columns: `stu_id, name, civics_class, date, reason, teacher_in_charge, teacher_email`.

| Question | Answer |
|----------|--------|
| PK before split? | **Composite `(stu_id, date)`** — one student can appear on many dates |
| Why not `stu_id` alone? | Same student ID repeats across rows |
| After 3NF — PK example | `Student.stu_id`, `Late.(stu_id, date)` or `Late.late_id` |
| FK example | `Late.stu_id` → `Student.stu_id` |
| Secondary key example | `Student.name`, `Student.civics_class` |

**3NF split (teacher depends on class, not student — do not leave teacher on Student):**
```text
Student(stu_id PK, name, civics_class FK)
CivicsClass(civics_class PK, teacher_name, teacher_email)
Late(stu_id FK, date, reason)   PK = (stu_id, date)  OR  late_id PK
```

**ER:** `CivicsClass` **1:M** `Student`; `Student` **1:M** `Late`.

### SQL vs NoSQL (one-liners)
- SQL: fixed schema, strong consistency, JOINs, good for related tables
- NoSQL (document): flexible schema, nested arrays/objects; related data via **embedding** or a **second query**
- Embedding when nested data is small; separate collections when nested data is large or deeply layered


<a id="sqlite"></a>
## 2. SQLite boilerplates

Exam loop: **connect → DROP/CREATE → INSERT from file → SELECT/UPDATE → commit → close**.

**Exam habits:**
- use **`?` placeholders** for values — never splice user input into SQL
- `conn.row_factory = sqlite3.Row` → rows behave like dicts (`row["name"]`)
- `INSERT OR IGNORE` skips PK clashes
- `fetchone()` / `fetchall()` pull results

**PK trap:** if a student can be late on many dates, `Late.stu_id` alone cannot be the only PK. Use composite `(date, stu_id)` or a surrogate `late_id`.


In [2]:
import sqlite3, csv

conn = sqlite3.connect("college.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()


cur.execute("DROP TABLE IF EXISTS Late")
cur.execute("DROP TABLE IF EXISTS Student")

cur.execute("""
CREATE TABLE IF NOT EXISTS Student(
    stu_id INTEGER PRIMARY KEY,
    name TEXT,
    civics_class TEXT,
    handphone INTEGER
)
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS Late(
    date TEXT,
    stu_id INTEGER,
    reason TEXT,
    PRIMARY KEY (date, stu_id),
    FOREIGN KEY (stu_id) REFERENCES Student(stu_id)
)
""")

with open("STUDENT.csv") as file:
    reader = csv.reader(file)
    next(reader)  # skip header
    for row in reader:
        cur.execute("""
            INSERT OR IGNORE INTO Student(stu_id, name, civics_class, handphone)
            VALUES (?,?,?,?)
        """, (row[0], row[1], row[2], row[3]))

with open("LATE.csv") as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        cur.execute("""
            INSERT OR IGNORE INTO Late(date, stu_id, reason)
            VALUES (?,?,?)
        """, (row[0], row[1], row[2]))

conn.commit()
print("tables created and seeded")


tables created and seeded


### CRUD template


In [4]:
# CRUD pattern (parameterised — never splice user input into SQL)
# CREATE
cur.execute("INSERT INTO Student(stu_id, name, civics_class, handphone) VALUES (?,?,?,?)",
            (25999, "Test Student", "25S02X", 81234567))

# READ
cur.execute("SELECT name, civics_class FROM Student WHERE stu_id = ?", (25999,))
for i in cur.fetchone():
    print(i)

# UPDATE
cur.execute("UPDATE Student SET name = ? WHERE stu_id = ?", ("Renamed Student", 25999))

# DELETE
cur.execute("DELETE FROM Student WHERE stu_id = ?", (25999,))
conn.commit()


Test Student
25S02X


<a id="sql"></a>
## 3. SQL query lookup

```sql
SELECT fields / aggregates
FROM table
     [INNER | LEFT OUTER] JOIN other ON pk = fk
WHERE condition          -- filter rows before grouping
GROUP BY field           -- one summary row per group
HAVING aggregate_cond    -- filter groups (not WHERE for COUNT/SUM)
ORDER BY field [DESC]
```

- `LIKE`: `name LIKE ?` with `"%Li%"` — wildcards around the search term
- **INNER JOIN** drops unmatched rows
- **LEFT OUTER JOIN** keeps every left-table row; missing right side is `NULL`
- After `GROUP BY`, you can only `SELECT` the grouped field(s) and aggregates — not a raw field like `Late.reason` unless you aggregate it (e.g. `MAX(reason)`)


In [8]:
# INNER JOIN: only students who have at least one late record
cur.execute("""
SELECT Student.name, Late.date, Late.reason
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
WHERE Student.civics_class = ?
ORDER BY Late.date
""", ("25S02X",))
#fetch all records in dictionary form from selection filter
for row in cur.fetchall():
    print(row['name'], row['date'], row['reason'])

# LEFT OUTER JOIN: keep students even if they have zero late records
cur.execute("""
SELECT Student.name, Late.date
FROM Student LEFT OUTER JOIN Late ON Student.stu_id = Late.stu_id
WHERE Student.civics_class = ?
""", ("25S02X",))
# names with date = None were never late


Sara Tay 2024-07-01 Alarm clock did not ring
Lee Xue Ying 2024-07-02 Heavy traffic
Dylan Koh 2024-07-03 Overslept
Chen Wei Li 2024-07-03 Missed the bus
Mohammad Rizwan 2024-07-04 Public transport delay
Nguyen Thi Mai 2024-07-05 Bus broke down
Anh Tran 2024-07-08 Alarm clock did not ring
Indira Devi 2024-07-09 Missed the bus
Kumar Suresh 2024-07-12 Bus broke down
Siti Nurhaliza 2024-07-14 Alarm clock did not ring
Nguyen Hoang 2024-07-15 Missed the bus
Prakash Ravi 2024-07-15 Overslept
Chen Wei Li 2024-07-15 Missed the bus
Yu Hao Ming 2024-07-16 Public transport delay
Ali Faisal 2024-07-17 Missed the bus
Tanisha Pillai 2024-07-18 Public transport delay
Philippe de la Cruz 2024-07-19 Missed the bus
Boon Kai Sheng 2024-07-22 Heavy traffic
Natasha Lim 2024-07-22 Bus arrived late
Lydia Tan 2024-07-23 Alarm clock did not ring
Chen Wei Li 2024-07-23 Missed the bus
Kumar Suresh 2024-07-26 Overslept
Nguyen Thi Mai 2024-07-28 Bus broke down
Indira Devi 2024-07-29 Public transport delay
Mohammad R

In [11]:
# Late count per civics_class
cur.execute("""
SELECT Student.civics_class, COUNT(*) AS total 
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
GROUP BY Student.civics_class;
""")
for row in cur.fetchall():
    print(row['civics_class'], row['total'])

# Same, but only classes with at least 3 late records
cur.execute("""
SELECT Student.civics_class, COUNT(*) AS total 
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
GROUP BY Student.civics_class 
HAVING COUNT(*) >= 3;
""")
for row in cur.fetchall():
    print(row['civics_class'], row['total'])

# Names containing "Li"
cur.execute("SELECT name FROM Student WHERE name LIKE ?", ("%Li%",))
for row in cur.fetchall():
    print(row["name"])


25S02X 67
25S02Y 90


[('25S02X', 67), ('25S02Y', 90)]

Ali Faisal
Lisa Lim
Liu Ying
Siti Nurhaliza
Kwan Li Mei
Philippe de la Cruz
Lim Wei Xiang
Amy Lim
Natasha Lim
Chen Wei Li


<a id="design"></a>
## 4. Design patterns

### Same student, same date, twice
Two morning/afternoon lates share the same `(stu_id, date)` → composite PK breaks.

**Fix:** add `late_id INTEGER PRIMARY KEY` (surrogate) and drop the composite PK, **or** add a third PK field (e.g. session/time) — surrogate is cleaner in exams.

### Repeat latecomers + never-late list


In [15]:
# Students with 2+ late records, sorted by count descending
cur.execute("""
SELECT Student.name, COUNT(*) AS latecount
FROM Student INNER JOIN Late ON Student.stu_id = Late.stu_id
GROUP BY Student.stu_id
HAVING COUNT(*) >= 2
ORDER BY latecount DESC
""")
for row in cur.fetchall():
    print(row["name"], row["latecount"])

# Prompt for class → students who were NEVER late
c = input("key in a class: ")
cur.execute("""
SELECT Student.name
FROM Student LEFT OUTER JOIN Late ON Student.stu_id = Late.stu_id
WHERE Late.stu_id IS NULL AND Student.civics_class = ?
""", (c,))
for row in cur.fetchall():
    print(row['name'])


key in a class: 25S02X
Tan Yu Wei
Sim Wen Jie
Rina Ong


<a id="mongo"></a>
## 5. NoSQL + pymongo *(prelim-only — parked for A-level P2)*

Document model: **database → collection → document (dict)**. National P2 papers use **SQLite**, not Mongo (~0% TYS P2). Keep this section as minimum recall if a prelim throws Mongo.

```text
client = MongoClient("mongodb://localhost:27017/")
db = client["name"]
coll = db["name"]
coll.delete_many({})
coll.insert_one(doc) / insert_many(list)
coll.find(filter, projection)
coll.update_one(filter, {"$set": {...}})   # filter FIRST
coll.drop() / client.drop_database("name")
```

**Join in exams:** second `find` with `$in` list of IDs, or embed related arrays inside one document. `$lookup` aggregation is extra — not required.

### Parked gaps (fix once, do not drill)
| Gap | Correct pattern |
|-----|-----------------|
| Projection | Use `_id`, not `id`. Inclusion only: `{"_id": 0, "Student_ID": 1, "Reason": 1}` |
| `update_one` | `update_one({"Student_ID": "25065", "Date": "2024-07-03"}, {"$set": {"Reason": "..."}})` |
| Embedding | Build nested list in Python → `insert_one`; query `{"lates": {"$ne": []}}` |
| `$lookup` | Not needed — `$in` + two queries is enough |


In [21]:
import pymongo, csv, json

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["college_db"]
students = db["student"]
lates = db["late"]
students.delete_many({})
lates.delete_many({})

with open("STUDENT.csv") as file:
    reader = csv.reader(file)
    headers = next(reader)
    for row in reader:
        students.insert_one(dict(zip(headers, row)))

with open("LATE.csv") as file:
    reader = csv.reader(file)
    headers = next(reader)
    for row in reader:
        lates.insert_one(dict(zip(headers, row)))

# find / projection / sort
for doc in students.find({"Class_Group": "25S02X"}, {"_id": 0, "Student_Name": 1, "Student_ID": 1}):
    print(doc)

# two-query "join" (exam-typical; no aggregation pipeline needed)
late_ids = lates.distinct("Student_ID")
for doc in students.find({"Student_ID": {"$in": late_ids}}, {"_id": 0, "Student_Name": 1}):
    print(doc["Student_Name"])

client.close()

# parked patterns — uncomment if prelim hits Mongo
# lates.find({"Date": {"$gt": "2024-07-02"}}, {"_id": 0, "Student_ID": 1, "Reason": 1})
# lates.update_one({"Student_ID": "25065", "Date": "2024-07-03"}, {"$set": {"Reason": "Updated reason"}})


{'Student_ID': '25032', 'Student_Name': 'Ali Faisal'}
{'Student_ID': '25064', 'Student_Name': 'Amy Lim'}
{'Student_ID': '25018', 'Student_Name': 'Anh Tran'}
{'Student_ID': '25023', 'Student_Name': 'Boon Kai Sheng'}
{'Student_ID': '25092', 'Student_Name': 'Chen Wei Li'}
{'Student_ID': '25065', 'Student_Name': 'Dylan Koh'}
{'Student_ID': '25033', 'Student_Name': 'Indira Devi'}
{'Student_ID': '25022', 'Student_Name': 'Kim Choon Yee'}
{'Student_ID': '25041', 'Student_Name': 'Kumar Suresh'}
{'Student_ID': '25039', 'Student_Name': 'Lee Xue Ying'}
{'Student_ID': '25054', 'Student_Name': 'Lydia Tan'}
{'Student_ID': '25087', 'Student_Name': 'Mohammad Rizwan'}
{'Student_ID': '25079', 'Student_Name': 'Natasha Lim'}
{'Student_ID': '25031', 'Student_Name': 'Nguyen Hoang'}
{'Student_ID': '25014', 'Student_Name': 'Nguyen Thi Mai'}
{'Student_ID': '25058', 'Student_Name': 'Philippe de la Cruz'}
{'Student_ID': '25088', 'Student_Name': 'Prakash Ravi'}
{'Student_ID': '25067', 'Student_Name': 'Rina Ong'}
{

<a id="cheatsheet"></a>
## 6. Exam cheat-sheet

### sqlite3 — minimum recall
```text
conn = sqlite3.connect("x.db"); conn.row_factory = sqlite3.Row; cur = conn.cursor()
CREATE TABLE (... PK, FK REFERENCES other(pk))
INSERT OR IGNORE INTO t(...) VALUES (?,?,?)
SELECT ... FROM a INNER JOIN b ON a.pk = b.fk WHERE ... ORDER BY ...
SELECT f, COUNT(*) FROM ... GROUP BY f HAVING COUNT(*) >= ?
UPDATE t SET field=? WHERE pk=?
DELETE FROM t WHERE pk=?
conn.commit(); conn.close()
csv: next(reader) then loop rows — never hardcode names in WHERE
LEFT JOIN + WHERE right.fk IS NULL  →  rows with no match on right
```

### pymongo — minimum recall *(prelim only)*
```text
coll.insert_one(dict) / insert_many(list)
coll.find({field: val}, {"_id": 0, field: 1})     # inclusion projection
coll.find({"n": {"$gte": 10, "$gt": d}})         # comparison operators
coll.update_one(filter_dict, {"$set": {field: val}})   # filter first
coll.delete_many({})
join: ids = lates.distinct("Student_ID"); students.find({"Student_ID": {"$in": ids}})
embed: parent["children"] = [{...}, ...]; coll.insert_one(parent)
```

### Theory — minimum recall
```text
PK unique + not null; FK references PK; composite = two+ fields together
1NF atomic → 2NF no partial dep on composite PK → 3NF no transitive dep
INNER = matches only; LEFT = keep all left rows
GROUP BY compresses rows; HAVING filters groups; WHERE filters rows before grouping
SQL vs NoSQL: schema fixed vs flexible; JOIN vs embed / second query
```
